In [6]:
import gradio as gr
from constants import guard_rail_message
from chatbot import run_chatbot, get_guard_rail
import asyncio


async def handler(user_message: str, history: list):
    stop_event = asyncio.Event()
    guard_result = {"allowed": None}

    async def run_guardrail():
        is_allowed = await get_guard_rail(user_message)
        guard_result["allowed"] = is_allowed
        if not is_allowed:
            stop_event.set()

    task = asyncio.create_task(run_guardrail())

    output = ""
    async for chunk in run_chatbot(user_message):
        if stop_event.is_set():
            yield guard_rail_message
            task.cancel()
            return

        output += chunk
        yield output

    # Edge case: guardrail finishes after the stream ends
    await task
    if guard_result["allowed"] is False:
        yield guard_rail_message


with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    textbox = gr.Textbox(placeholder="Type a message...")

    textbox.submit(
        fn=handler,
        inputs=[textbox, chatbot],
        outputs=chatbot,
    )

demo.launch()

Task exception was never retrieved
future: <Task finished name='Task-229' coro=<handler.<locals>.run_guardrail() done, defined at C:\Users\stane\AppData\Local\Temp\ipykernel_27020\1848404352.py:11> exception=InvalidUpdateError('Expected dict, got ce se intampla daca fur \nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE')>
Traceback (most recent call last):
  File "C:\Users\stane\AppData\Local\Temp\ipykernel_27020\1848404352.py", line 12, in run_guardrail
    is_allowed = await get_guard_rail(user_message)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\stane\Desktop\cod\ChatBot\chatbot.py", line 50, in get_guard_rail
    result = await guard_rail_agent.ainvoke(question)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\stane\Desktop\cod\ChatBot\.venv\Lib\site-packages\langgraph\pregel\main.py", line 3182, in ainvoke
    async for chunk in self.astream(
  File "C:\Users\stane\Des

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
